Alright I got cellphone DB to work, now I need to do differential communicaton testing, 

My idea is unfortunately I have to loop through each individual and run cpdb_degs_analysis_method. Then, I collect all of the means.txt values, add sex, and then I'm going to have to loop through a lot of shit. I think its going to have to be a cluster by cluster, interaction by interation nested for loop where I coalesce the mean from each individual and run a linear regression. Before I do that, I will need to make sure it has a reasonable distribution and doesnt need some kind of special regression or non-parametric test. 

In [62]:
#libraries
import anndata  
import pandas as pd
import anndata as ad
import seaborn as sb
import scanpy as sc
import cellphonedb
import glob
import os
import sys

In [63]:
full_object = anndata.io.read_h5ad('C:/Users/Gabe/Desktop/RNA_object_human_names_anndata.h5ad')


In [64]:
#create an example subset object
#idk why this works in dot format but not 
subset_object =full_object[full_object.obs.individual=='T17D']

In [65]:
#t17d is the only one there
print(subset_object.obs.individual)

print(full_object.obs.shape)
print(subset_object.obs.shape)
### looks like it worked to me

T17D_AAACAGCCAAGGCCAA-1    T17D
T17D_AAACATGCAAGCTTAT-1    T17D
T17D_AAACATGCACAATGCC-1    T17D
T17D_AAACATGCAGCATGGA-1    T17D
T17D_AAACCAACAAAGGCCA-1    T17D
                           ... 
T17D_TTTGTGGCATTGTCCT-1    T17D
T17D_TTTGTGTTCAACAAGG-1    T17D
T17D_TTTGTGTTCATAAGCC-1    T17D
T17D_TTTGTTGGTGCCTCAC-1    T17D
T17D_TTTGTTGGTTTCGCCA-1    T17D
Name: individual, Length: 2795, dtype: object
(52517, 77)
(2795, 77)


Ok, the first thing I am going to do is write a loop that makes subset_objects

In [66]:
objects = {}
k =0
for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    print(i)
    key =  f"{'obj_'}{i}" #dynamically name object
    value = full_object[full_object.obs.individual==i] #subset object
    objects[key] = value 
    k += 1



T13D
A12D
T5D
A22D
B23F
C15M
T17D
C21D
B21M
B23M
C14F
T19D
T14D
C13F
C13M
T11D
B21F
D4F
D4M
C15F
A25D
C14M


In [67]:
#I think it worked
objects['obj_A12D'].obs.individual

A12D_AAACAGCCAAACTAAG-1    A12D
A12D_AAACAGCCAAAGCTAA-1    A12D
A12D_AAACAGCCATTAGGTT-1    A12D
A12D_AAACATGCAAACGGGC-1    A12D
A12D_AAACCGAAGAACCTGT-1    A12D
                           ... 
A12D_TTTGTGTTCACGCGGT-1    A12D
A12D_TTTGTGTTCTACCTGC-1    A12D
A12D_TTTGTTGGTCGTTACT-1    A12D
A12D_TTTGTTGGTCTAACAG-1    A12D
A12D_TTTGTTGGTTGAGGTC-1    A12D
Name: individual, Length: 3993, dtype: object

Ok, next, I want to loop through the cellphone function, I will need to make sure to also make a dynamic path just to stop any mess in my folder

Wait a second, will it work with the URL meta data paths etc? I think because everything I care about is in the anndata object, I can write the relevant files at each subset iteration

In [ ]:
### SUCCEEDED, DONT NEED TO RUN AGAIN
## Write files
import hdf5plugin
from scipy.sparse import csr_matrix

for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    key =  f"{'obj_'}{i}" #dynamically name object
    relevant_object = objects[key]
    temp_meta_data = relevant_object.obs[['cell_barcode', 'harmony.wnn_res0.4_clusters']]
    
    cells = {
    "Cell" : temp_meta_data['cell_barcode'],
    "harmony.wnn_res0.4_clusters" : temp_meta_data['harmony.wnn_res0.4_clusters']
    
    }
    cells = pd.DataFrame(cells)
    cells2 = cells[['Cell','harmony.wnn_res0.4_clusters']]

    filename = f"{"A:/CellPhoneDB 030225/"}{i}{"/cluster_labels_metadata.tsv"}"
    # I need to add a line here to make the folder
    
    file_path = os.makedirs(f"{"A:/CellPhoneDB 030225/"}{i}{"/"}")

    cells2.to_csv(filename, sep="\t") 

    folder_path = f"{"A:/CellPhoneDB 030225/"}{i}{"/normalised_log_counts.h5ad"}"

    relevant_object.X = csr_matrix(relevant_object.X)
    relevant_object.write_h5ad(folder_path)


#### Succeeded, dont need to run again

c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: Future

In [71]:
for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    key =  f"{'obj_'}{i}" #dynamically name object
    relevant_object = objects[key]
    temp_meta_data = relevant_object.obs[['cell_barcode', 'harmony.wnn_res0.4_clusters']]
    
    cells = {
    "Cell" : temp_meta_data['cell_barcode'],
    "cell_type" : temp_meta_data['harmony.wnn_res0.4_clusters']
    
    }
    cells = pd.DataFrame(cells)
    cells2 = cells[['Cell','cell_type']]

    filename = f"{"A:/CellPhoneDB 030225/"}{i}{"/cluster_labels_metadata2.tsv"}"
    # I need to add a line here to make the folder
    
    cells2.to_csv(filename, sep="\t") 


In [ ]:
 from cellphonedb.src.core.methods import cpdb_degs_analysis_method

for i in set(full_object.obs.individual):
    cpdb_file_path = 'A:/CellPhoneDB 030225/v5.0.0/cellphonedb.zip' # Dont change
    degs_file_path = 'A:/CellPhoneDB 030225/human_named_DEGs.tsv' # I think this needs to be edited

    meta_file_path =  f"{"A:/CellPhoneDB 030225/"}{i}{"/cluster_labels_metadata3.tsv"}"
    counts_file_path  =  f"{"A:/CellPhoneDB 030225/"}{i}{"/normalised_log_counts.h5ad"}"
    out_path =  f"{"A:/CellPhoneDB 030225/"}{i}"

    cpdb_results = cpdb_degs_analysis_method.call(
        cpdb_file_path = cpdb_file_path,                            # mandatory: CellphoneDB database zip file.
        meta_file_path = meta_file_path,                            # mandatory: tsv file defining barcodes to cell label.
        counts_file_path = counts_file_path,                        # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
        degs_file_path = degs_file_path,                            # mandatory: tsv file with DEG to account.
        counts_data = 'gene_name',                                # defines the gene annotation in counts matrix.
        score_interactions = True,                                  # optional: whether to score interactions or not. 
        threshold = 0.1,                                            # defines the min % of cells expressing a gene for this to be employed in the analysis.
        result_precision = 3,                                       # Sets the rounding for the mean values in significan_means.
        separator = '_',                                            # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
        debug = False,                                              # Saves all intermediate tables emplyed during the analysis in pkl format.
        output_path = out_path,                                     # Path to save results
        output_suffix = None,                                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
        threads = 25
        ) 

[ ][CORE][04/03/25-14:41:27][INFO] [Cluster DEGs Analysis] Threshold:0.1 Precision:3


MissingRequiredArgumentsException: All of the following arguments need to be provided: cpdb_file_path, meta_file_path, counts_file_path, degs_file_path, counts_data, output_path

In [ ]:
meta =pd.read_csv('A:/CellPhoneDB 030225/human_named_meta.tsv', sep= '\t')
meta.head()

FileNotFoundError: [Errno 2] No such file or directory: 'A:/CellPhoneDB 030225/human_named_meta2.tsv'

In [74]:
meta =pd.read_csv(meta_file_path, sep= '\t')
meta.head()

,Unnamed: 0,Cell,cell_type
0,T13D_AAACAGCCATGAGCAG-1,T13D_AAACAGCCATGAGCAG-1,23
1,T13D_AAACCAACAGGAACCA-1,T13D_AAACCAACAGGAACCA-1,4
2,T13D_AAACCAACATGTCGCG-1,T13D_AAACCAACATGTCGCG-1,3
3,T13D_AAACCGAAGAGAAGGG-1,T13D_AAACCGAAGAGAAGGG-1,7
4,T13D_AAACCGAAGCTCCTTA-1,T13D_AAACCGAAGCTCCTTA-1,7


In [77]:
# Create properly formatted metadata file
import pandas as pd

for i in set(full_object.obs.individual):
    # Extract metadata for this individual
    subset = full_object[full_object.obs.individual == i]

    if i == 'GH':
        continue
    
    # Create metadata DataFrame with required format
    metadata = pd.DataFrame({
        'Cell': subset.obs.index,  # Cell barcodes as index
        'cell_type': subset.obs['harmony.wnn_res0.4_clusters'].astype(str)  # Cell type annotations
    })
    
    # Save metadata with proper format
    meta_file_path = f"A:/CellPhoneDB 030225/{i}/cluster_labels_metadata3.tsv"
    metadata.to_csv(meta_file_path, sep='\t', index=True)


In [78]:
 f"A:/CellPhoneDB 030225/{i}/cluster_labels_metadata3.tsv"

'A:/CellPhoneDB 030225/C14M/cluster_labels_metadata3.tsv'